## 01 Test the access to the RSP APIwith pyvo 

- author : Sylvie Dagoret-Campagne
- creation date : 2026-02-28

In [ ]:
import os
import pyvo
import requests

In [ ]:
# 1. Token d'authentification (stocké dans une variable d'environnement)
# export RSP_TOKEN="votre_token" dans ton .zshrc ou .bashrc
token = os.environ["RSP_TOKEN"]
#print(f"token {token}")

In [ ]:
# Méthode recommandée avec pyvo récent
credential = pyvo.auth.authsession.AuthSession()
credential.credentials.set("lsst-token", token)

# 2. Session HTTP authentifiée
session = requests.Session()
session.headers["Authorization"] = f"Bearer {token}"

service = pyvo.dal.TAPService("https://data.lsst.cloud/api/tap", session=session)

In [ ]:
# 3. Connexion au service TAP de Rubin
rsp_tap_url = "https://data.lsst.cloud/api/tap"
service = pyvo.dal.TAPService(rsp_tap_url, session=session)

In [ ]:
# Liste uniquement les noms (plus léger, évite le bug)
table_names = list(service.tables.keys())
for name in table_names:
    print(name)

In [ ]:
# 4. Requête ADQL (exemple : visites en bandes g et r autour de ECDFS)
query = """
SELECT visit, ra, dec, band, expMidptMJD
FROM dp1.Visit
WHERE CONTAINS(
      POINT('ICRS', ra, dec),
      CIRCLE('ICRS', 53.13, -28.10, 3)
    ) = 1
  AND band IN ('g', 'r')
ORDER BY expMidptMJD ASC
"""

In [ ]:
# 5. Soumission et exécution asynchrone du job
job = service.submit_job(query)
job.run()
job.wait(phases=["COMPLETED", "ERROR"])
print("Job phase:", job.phase)
if job.phase == "ERROR":
    job.raise_if_error()

In [ ]:
# 6. Récupération des résultats en table Astropy
results = job.fetch_result().to_table()
print(f"Nombre de lignes : {len(results)}")
print(results[:5])